In [ ]:
!pip install segmentation-models-pytorch --quiet

In [ ]:

import gradio as gr
import torch
import numpy as np
import cv2
from segmentation_models_pytorch import Unet

# -------------------------
# CONFIGURACIÓN
# -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "/content/drive/MyDrive/MisModelos/best_unet_resnet34.pth"

# -------------------------
# DEFINIR MODELO
# -------------------------
model = Unet(encoder_name="resnet34", in_channels=4, classes=3)  # 4 canales: R,G,B limpio + Green Raw

# Cargar checkpoint
checkpoint = torch.load(model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

# -------------------------
# FUNCIÓN DE LIMPIEZA
# -------------------------
def clean_fundus_image(img):
    kernel_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    kernel_large = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    img_closed = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel_small)
    img_closed = cv2.morphologyEx(img_closed, cv2.MORPH_CLOSE, kernel_large)
    img_blurred = cv2.GaussianBlur(img_closed, (3, 3), 0)
    background = cv2.GaussianBlur(img_blurred, (31, 31), 0)
    img_normalized = cv2.subtract(img_blurred, background)
    img_normalized = cv2.add(img_normalized, 128)
    img_normalized = np.clip(img_normalized, 0, 255).astype(np.uint8)
    # CLAHE
    lab = cv2.cvtColor(img_normalized, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge((l, a, b))
    img_final = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    return img_final

# -------------------------
# CALCULAR CDR (robusto con bounding box)
# -------------------------
def calculate_cdr(mask):
    def get_combined_bbox_height(contours):
        if len(contours) == 0:
            return 0.0

        min_y = float('inf')
        max_y = float('-inf')

        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            min_y = min(min_y, y)
            max_y = max(max_y, y + h)

        if min_y == float('inf') or max_y == float('-inf'):
            return 0.0

        return max_y - min_y

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

    # Disco
    binary_disc_mask = (mask==1).astype(np.uint8)
    cleaned_binary_disc_mask = cv2.morphologyEx(binary_disc_mask, cv2.MORPH_OPEN, kernel, iterations=1)
    contours_disc, _ = cv2.findContours(cleaned_binary_disc_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    h_disc = get_combined_bbox_height(contours_disc)

    # Copa
    binary_cup_mask = (mask==2).astype(np.uint8)
    cleaned_binary_cup_mask = cv2.morphologyEx(binary_cup_mask, cv2.MORPH_OPEN, kernel, iterations=1)
    contours_cup, _ = cv2.findContours(cleaned_binary_cup_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    h_cup = get_combined_bbox_height(contours_cup)

    # Mostrar información de depuración
    print("=== CDR DEBUG ===")
    print(f"Disco: {len(contours_disc)} contornos, altura total h_disc={h_disc}")
    print(f"Copa: {len(contours_cup)} contornos, altura total h_cup={h_cup}")

    if h_disc == 0:
        return 0.0

    cdr = h_cup / h_disc
    print(f"CDR calculado: {cdr:.3f}")
    return float(cdr)

# -------------------------
# FUNCIÓN DE PREDICCIÓN
# -------------------------
# -------------------------
# FUNCIÓN DE PREDICCIÓN CON CDR CORRECTO
# -------------------------
def predict(image):
    # 1. Convertir a RGB y redimensionar
    img_raw = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img_raw = cv2.resize(img_raw, (512, 512))

    # 2. Imagen limpia
    img_cleaned = clean_fundus_image(img_raw)

    # 3. Canal verde sin limpiar
    green_channel = img_raw[:, :, 1]
    green_expanded = np.expand_dims(green_channel, axis=-1)

    # 4. Concatenar a 4 canales y normalizar
    img_4ch = np.concatenate([img_cleaned, green_expanded], axis=-1).astype(np.float32) / 255.0
    tensor = torch.tensor(img_4ch.transpose(2,0,1)).unsqueeze(0).to(device)

    # 5. Inferencia
    with torch.no_grad():
        logits = model(tensor)
        pred_mask = torch.argmax(logits, dim=1).cpu().numpy()[0]

    # 6. Calcular CDR usando lógica de compute_CDR_info
    def compute_CDR_info(mask):
        disc = (mask == 1).astype(np.uint8)
        cup  = (mask == 2).astype(np.uint8)

        def height(m):
            ys = np.where(m > 0)[0]
            if len(ys) == 0:
                return 0
            return ys.max() - ys.min() + 1

        h_disc = height(disc)
        h_cup  = height(cup)

        if h_disc == 0:
            return 0.0, 0, 0  # CDR, h_disc, h_cup

        h_cup_clamped = min(h_cup, h_disc)
        CDR = h_cup_clamped / h_disc
        return CDR, h_disc, h_cup

    cdr, h_disc, h_cup = compute_CDR_info(pred_mask)

    diagnosis = f"⚠ Sospecha de glaucoma (CDR={cdr:.3f})" if cdr > 0.6 else f"Normal (CDR={cdr:.3f})"

    # 7. Máscara coloreada
    colored_mask = np.zeros((512, 512, 3), dtype=np.uint8)
    colored_mask[pred_mask == 1] = (0, 255, 0)  # Disco
    colored_mask[pred_mask == 2] = (255, 0, 0)  # Copa

    # 8. Info de depuración
    debug_info = f"h_disc={h_disc}, h_cup={h_cup}, CDR={cdr:.3f}"

    return colored_mask, diagnosis, debug_info


# -------------------------
# INTERFAZ GRADIO
# -------------------------
demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="numpy", label="Sube una imagen de fondo de ojo"),
    outputs=[
        gr.Image(type="numpy", label="Máscara Predicha"),
        gr.Textbox(label="Diagnóstico / CDR"),
        gr.Textbox(label="Info")  # aquí verás los valores de h_disc y h_cup
    ],
    title="Detector de Glaucoma (Segmentación + CDR)",
    description="Carga una imagen de fondo de ojo y el modelo segmenta disco/copa y calcula el CDR.",
    allow_flagging="never"
)


demo.launch(share=True, inline=False)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
model_path = "/content/drive/MyDrive/MisModelos/best_unet_resnet34.pth"


In [ ]:
!ls "/content/drive/MyDrive/MisModelos"


best_unet_resnet34.pth
